# 03 · Record validation without losing evidence

Normalize only according to explicit policy. Retain original values beside casts and normalized columns. We reject all occurrences of a duplicated order key; selecting an arbitrary survivor would hide a producer defect.


## Environment
Upload the prepared sample files before the session, then use notebook 00 to check the configured storage. This notebook then runs independently, top to bottom. Spark 3.5 is the target; no Hive catalog is used. Set `BASE_PATH` in the following cell or set `DQ_BASE_PATH` in the driver environment.


In [ ]:
import os
import uuid

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("OrderDataQuality").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.shuffle.partitions", "4")  # tiny teaching datasets only
spark.conf.set("spark.sql.ansi.enabled", "true")
spark.conf.set("spark.sql.csv.parser.columnPruning.enabled", "false")
BASE_PATH = os.environ.get(
    "DQ_BASE_PATH", "s3://YOUR-BUCKET/training/order-quality"
).rstrip("/")
# On EMR/Glue edit the default above if driver environment variables are unavailable.
# Spark VM: hdfs:///user/student/order-quality ; local: file:///tmp/order-quality
assert (
    "YOUR-BUCKET" not in BASE_PATH
), "Set DQ_BASE_PATH or edit BASE_PATH before running"
PROCESSING_DATE = os.environ.get("DQ_PROCESSING_DATE", "2024-01-03")
RUN_ID = uuid.uuid4().hex
RAW_PATH, BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUARANTINE_PATH, AUDIT_PATH = [
    f"{BASE_PATH}/{layer}"
    for layer in ["raw", "bronze", "silver", "gold", "quarantine", "audit"]
]
print("Spark", spark.version, "storage", BASE_PATH, "run", RUN_ID)


## Inspect the source
Each JSON line is one order envelope. Preserve the original text and source filename before parsing so even corrupt lines remain accountable.


In [ ]:
ORDER_FIELDS = [
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "unit_price",
    "discount",
    "status",
    "order_date",
    "shipped_date",
    "cancellation_reason",
    "order_total",
    "seller_id",
    "country",
    "arrival_date",
]
order_schema = T.StructType(
    [T.StructField(c, T.StringType(), True) for c in ORDER_FIELDS]
    + [T.StructField("_corrupt_record", T.StringType(), True)]
)


def read_orders(path):
    # Preserve one source envelope per physical JSON line, including malformed JSON.
    # Source row IDs are materialized before branching; raw_text supports replay.
    raw = (
        spark.read.text(path)
        .withColumnRenamed("value", "raw_text")
        .withColumn("source_file", F.input_file_name())
        .withColumn("source_row_id", F.monotonically_increasing_id())
    )
    parsed = raw.withColumn(
        "parsed",
        F.from_json(
            "raw_text",
            order_schema,
            {"mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt_record"},
        ),
    )
    return parsed.select("source_row_id", "source_file", "raw_text", "parsed.*").cache()


orders = read_orders(f"{RAW_PATH}/orders")
orders.count()  # materialize once before splitting
customers = spark.read.option("header", True).csv(f"{RAW_PATH}/customers")
products = spark.read.option("header", True).csv(f"{RAW_PATH}/products")
items = spark.read.option("header", True).csv(f"{RAW_PATH}/order_items")
orders.show(30, truncate=False)


## Safe conversion with ANSI enabled
`try_cast` returns null for a failed conversion. Compare original and converted values to distinguish missing input from an invalid supplied value. Decimal currency avoids binary floating-point arithmetic.


In [ ]:
typed = (
    orders.withColumn("customer_id_clean", F.trim("customer_id"))
    .withColumn("status_clean", F.upper(F.trim("status")))
    .withColumn("qty", F.expr("try_cast(quantity as int)"))
    .withColumn("price", F.expr("try_cast(unit_price as decimal(18,2))"))
    .withColumn("discount_value", F.expr("try_cast(discount as decimal(8,2))"))
    .withColumn("total", F.expr("try_cast(order_total as decimal(18,2))"))
    .withColumn("event_date", F.expr("try_cast(order_date as date)"))
    .withColumn("shipped_on", F.expr("try_cast(shipped_date as date)"))
    .withColumn("arrived_on", F.expr("try_cast(arrival_date as date)"))
)
# Reference tables are deduplicated for membership joins, not as a silent repair.
# A separate customer-key check still exposes duplicate reference records.
customer_keys = (
    customers.select(F.col("customer_id").alias("customer_id_clean"))
    .distinct()
    .withColumn("known_customer", F.lit(True))
)
product_keys = (
    products.select("product_id").distinct().withColumn("known_product", F.lit(True))
)
typed = (
    typed.join(customer_keys, "customer_id_clean", "left")
    .join(product_keys, "product_id", "left")
    .withColumn("key_count", F.count("*").over(Window.partitionBy("order_id")))
)

typed.select(
    "order_id",
    "quantity",
    "qty",
    (F.col("quantity").isNotNull() & F.col("qty").isNull()).alias("cast_failed"),
).show(30)


## Completeness: null, empty and whitespace
Completeness uses all source envelopes as its denominator, including corrupt input. State your denominator; a metric over parsed-only data answers a different question.


In [ ]:
missing = F.col("customer_id").isNull() | (F.length(F.trim("customer_id")) == 0)
orders.select(
    F.count("*").alias("rows"),
    F.sum(missing.cast("int")).alias("missing"),
    (100 * F.avg((~missing).cast("double"))).alias("completeness_percentage"),
).show()
orders.filter(missing).select("order_id", "customer_id").show()
orders.filter(~missing).select("order_id", "customer_id").show()


## Exact duplicates versus duplicate business keys
Exact duplicates match every business field. Conflicting copies share an order ID but differ elsewhere. Ignore lineage fields when detecting exact duplicates; they intentionally differ per source occurrence.


In [ ]:
orders.groupBy(*ORDER_FIELDS).count().filter("count > 1").show(truncate=False)
duplicates = orders.groupBy("order_id").count().filter("count > 1")
orders.join(duplicates.select("order_id"), "order_id").orderBy("order_id").show(
    truncate=False
)
customers.groupBy("customer_id").count().filter("count > 1").show()


## Formats and lengths
These pragmatic regex checks are teaching contracts, not universal email/telephone validators. Product codes must match SKU plus three digits; names must contain 2–80 trimmed characters.


In [ ]:
customer_checks = (
    customers.withColumn(
        "email_ok",
        F.coalesce(F.col("email").rlike(r"^[^\s@]+@[^\s@]+\.[^\s@]+$"), F.lit(False)),
    )
    .withColumn(
        "phone_ok",
        F.coalesce(F.col("phone").rlike(r"^\+[1-9][0-9]{7,14}$"), F.lit(False)),
    )
    .withColumn(
        "name_ok",
        F.coalesce(F.length(F.trim("customer_name")).between(2, 80), F.lit(False)),
    )
)
customer_checks.filter("email_ok and phone_ok and name_ok").show()
customer_checks.filter("not (email_ok and phone_ok and name_ok)").show()
products.withColumn(
    "code_ok",
    F.col("product_code").rlike("^SKU-[0-9]{3}$") & (F.length("product_code") == 7),
).show()
typed.select(
    "order_id",
    F.coalesce(F.col("order_id").rlike("^O[0-9]{3}$"), F.lit(False)).alias("id_ok"),
).show()


## Dates and late arrivals
Invalid dates, future dates, old events and out-of-window events are distinct findings. Our batch contract is yesterday only. A valid historical event can still be late; a backfill should use a different explicit window.


In [ ]:
typed.select(
    "order_id",
    "order_date",
    "event_date",
    F.col("event_date").isNull().alias("invalid_date"),
    (F.col("event_date") > F.to_date(F.lit(PROCESSING_DATE))).alias("future"),
    (F.col("event_date") < F.date_sub(F.to_date(F.lit(PROCESSING_DATE)), 365)).alias(
        "old"
    ),
    (F.datediff("arrived_on", "event_date") > 1).alias("late"),
    (F.col("event_date") != F.date_sub(F.to_date(F.lit(PROCESSING_DATE)), 1)).alias(
        "outside_window"
    ),
).show(30)


## Foreign keys are still your responsibility
Spark file storage does not enforce relational foreign keys. `left_anti` returns records with no matching reference; null and blank keys appear here too. Deduplicate reference keys first for joins that attach attributes to avoid multiplying orders.


In [ ]:
typed.join(
    customers.select(F.col("customer_id").alias("customer_id_clean")).distinct(),
    "customer_id_clean",
    "left_anti",
).select("order_id", "customer_id_clean").show()
typed.join(products.select("product_id").distinct(), "product_id", "left_anti").select(
    "order_id", "product_id"
).show()


## Multiple failures per record
Range, domain, uniqueness, referential and business predicates below share a transparent representation. Each invalid source occurrence stays one row with an array of failures; explode only for rule reports, never for source-count reconciliation.


In [ ]:
# A predicate means PASS. NULL is a failure unless the rule explicitly permits it.
rules = [
    (
        "DQ000",
        "parseable",
        "raw_text",
        "Malformed JSON",
        F.col("_corrupt_record").isNull() & F.col("order_id").isNotNull(),
    ),
    (
        "DQ001",
        "customer present",
        "customer_id",
        "Missing or blank customer",
        F.length("customer_id_clean") > 0,
    ),
    (
        "DQ002",
        "positive integer quantity",
        "quantity",
        "Not an integer in 1..1000",
        F.col("qty").between(1, 1000),
    ),
    (
        "DQ003",
        "nonnegative price",
        "unit_price",
        "Invalid or negative price",
        F.col("price") >= 0,
    ),
    (
        "DQ004",
        "allowed status",
        "status",
        "Unknown status",
        F.col("status_clean").isin("CREATED", "PAID", "SHIPPED", "CANCELLED"),
    ),
    (
        "DQ005",
        "unique order key",
        "order_id",
        "Duplicate business key; quarantine all copies",
        F.col("key_count") == 1,
    ),
    (
        "DQ006",
        "event window",
        "order_date",
        "Invalid, future, old or outside daily window",
        F.col("event_date") == F.date_sub(F.to_date(F.lit(PROCESSING_DATE)), 1),
    ),
    (
        "DQ007",
        "known customer",
        "customer_id",
        "Customer reference not found",
        F.coalesce(F.col("known_customer"), F.lit(False)),
    ),
    (
        "DQ008",
        "known product",
        "product_id",
        "Product reference not found",
        F.coalesce(F.col("known_product"), F.lit(False)),
    ),
    (
        "DQ009",
        "discount range",
        "discount",
        "Discount outside 0..100",
        F.col("discount_value").between(0, 100),
    ),
    (
        "BR001",
        "shipment date",
        "shipped_date",
        "SHIPPED requires valid shipped_date",
        (F.col("status_clean") != "SHIPPED") | F.col("shipped_on").isNotNull(),
    ),
    (
        "BR002",
        "cancellation reason",
        "cancellation_reason",
        "CANCELLED requires reason",
        (F.col("status_clean") != "CANCELLED")
        | (F.length(F.trim("cancellation_reason")) > 0),
    ),
    (
        "BR003",
        "nonnegative total",
        "order_total",
        "Invalid or negative order total",
        F.col("total") >= 0,
    ),
    (
        "BR004",
        "order arithmetic",
        "order_total",
        "Header differs from quantity times unit price",
        F.abs(F.col("total") - F.col("qty") * F.col("price"))
        <= F.lit("0.01").cast("decimal(18,2)"),
    ),
]
failure_structs = [
    F.when(
        ~F.coalesce(predicate, F.lit(False)),
        F.struct(
            F.lit(rule_id).alias("dq_rule_id"),
            F.lit(name).alias("dq_rule_name"),
            F.lit(column).alias("dq_column"),
            F.lit(reason).alias("dq_reason"),
        ),
    )
    for rule_id, name, column, reason, predicate in rules
]
scored = (
    typed.withColumn(
        "dq_failures", F.filter(F.array(*failure_structs), lambda x: x.isNotNull())
    )
    .withColumn(
        "dq_status", F.when(F.size("dq_failures") == 0, "VALID").otherwise("INVALID")
    )
    .withColumn("pipeline_run_id", F.lit(RUN_ID))
    .withColumn("processing_timestamp", F.current_timestamp())
    .cache()
)
scored.count()
valid = scored.filter("dq_status = 'VALID'")
rejected = scored.filter("dq_status = 'INVALID'")
valid.select("order_id", "quantity", "status", "dq_status").show(truncate=False)
rejected.select("order_id", "source_row_id", "dq_failures").show(30, truncate=False)


## Quarantine is an output
Reject means exclude from the accepted path. Quarantine retains the record and reasons for diagnosis/replay. This is a dead-letter-like dataset; Spark batch storage itself does not provide a message queue’s retry semantics.


In [ ]:
valid.write.mode("errorifexists").parquet(f"{SILVER_PATH}/record_lab/{RUN_ID}")
rejected.write.mode("errorifexists").parquet(f"{QUARANTINE_PATH}/record_lab/{RUN_ID}")
rejected.select(
    "source_row_id",
    "source_file",
    "pipeline_run_id",
    "processing_timestamp",
    F.explode("dq_failures").alias("failure"),
).select("*", "failure.*").show(50, truncate=False)
assert orders.count() == valid.count() + rejected.count()
